# Finanzas Públicas del Perú — Series Históricas BCRP

Este notebook descarga series anuales desde la API del **BCRPData** y las exporta como una nueva hoja (`C-4_Historico`) en el archivo `datos.xlsx`.

**Series extraídas (todas en % del PBI, frecuencia anual):**

| Código | Serie | Disponible desde |
|---|---|---|
| PM05780FA | Resultado Económico SPNF | 1970 |
| PM05776FA | Resultado Primario SPNF | 1970 |
| PM10083FA | Intereses SPNF | 1970 |
| PM10080FA | Gastos de Capital SPNF | 1970 |
| PM10081FA | Inversión Pública SPNF | 1970 |
| PM10117FA | Total Ingresos Corrientes GC | 1970 |
| PM10103FA | Ingresos Tributarios GC | 1970 |
| PM10104FA | Impuesto a los Ingresos / IR | 1970 |
| PM10108FA | IGV | 1970 |
| PM10111FA | ISC | 1970 |
| PM10116FA | Ingresos No Tributarios GC | 1970 |
| PM10188FA | Deuda Pública Total | 1999 |
| PM10189FA | Deuda Pública Externa | 1999 |
| PM10197FA | Deuda Pública Interna | 1999 |
| PM10213FA | Bonos Soberanos | 1999 |

**Fuente:** BCRPData — https://estadisticas.bcrp.gob.pe

In [ ]:
# ─── 0. Instalación de dependencias (solo si es necesario) ───────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

for pkg in ['requests', 'pandas', 'openpyxl', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        install(pkg)
        
print('Dependencias listas.')

In [ ]:
# ─── 1. Imports ──────────────────────────────────────────────────────────────
import requests
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from openpyxl import load_workbook
import time

print('Imports OK')

In [ ]:
# ─── 2. Configuración ────────────────────────────────────────────────────────

# Ruta al archivo Excel principal (ajustar si es necesario)
DATA_PATH = Path('../data/datos.xlsx')

# Período de consulta
YEAR_INI = 2000   # Cambia a 1970 si quieres toda la serie disponible
YEAR_FIN = 2025

# Base URL de la API del BCRP
API_BASE = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api'

# ── Diccionario de series a descargar ──
# Formato: {codigo_bcrp: nombre_columna_legible}
SERIES = {
    # --- Resultado fiscal SPNF (% PBI) ---
    'PM05780FA': 'Resultado Económico SPNF',
    'PM05776FA': 'Resultado Primario SPNF',
    'PM10083FA': 'Intereses SPNF',
    'PM10080FA': 'Gastos de Capital SPNF',
    'PM10081FA': 'Inversión Pública SPNF',
    # --- Ingresos Gobierno Central (% PBI) ---
    'PM10117FA': 'Total Ingresos Corrientes GC',
    'PM10103FA': 'Ingresos Tributarios GC',
    'PM10104FA': 'Impuesto a la Renta (IR)',
    'PM10108FA': 'IGV',
    'PM10111FA': 'ISC',
    'PM10116FA': 'Ingresos No Tributarios GC',
    # --- Deuda Pública (% PBI, desde 1999) ---
    'PM10188FA': 'Deuda Pública Total',
    'PM10189FA': 'Deuda Pública Externa',
    'PM10197FA': 'Deuda Pública Interna',
    'PM10213FA': 'Bonos Soberanos',
}

print(f'Total de series a descargar: {len(SERIES)}')
print(f'Período: {YEAR_INI} – {YEAR_FIN}')

In [ ]:
# ─── 3. Función de descarga ───────────────────────────────────────────────────

def fetch_serie(codigo: str, year_ini: int, year_fin: int, retries: int = 3) -> pd.Series:
    """
    Descarga una serie anual desde la API del BCRP en formato JSON.
    Retorna un pd.Series indexado por año (int).
    """
    url = f'{API_BASE}/{codigo}/json/{year_ini}/{year_fin}/esp'
    
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            data = resp.json()
            
            # La API devuelve: {"config": {...}, "periods": [{"name": "2000", "values": ["val"]}, ...]}
            periods = data.get('periods', [])
            records = {}
            for p in periods:
                year = int(p['name'])
                raw  = p['values'][0]          # primer (y único) valor
                try:
                    records[year] = float(raw)
                except (ValueError, TypeError):
                    records[year] = float('nan')  # 'n.d.' u otros valores no numéricos
            
            return pd.Series(records, name=codigo)
        
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
            else:
                print(f'  ⚠️  Error descargando {codigo}: {e}')
                return pd.Series(name=codigo, dtype=float)

print('Función fetch_serie definida.')

In [ ]:
# ─── 4. Descarga de todas las series ─────────────────────────────────────────

all_series = {}

for codigo, nombre in SERIES.items():
    print(f'  Descargando: {nombre} ({codigo})...')
    s = fetch_serie(codigo, YEAR_INI, YEAR_FIN)
    all_series[nombre] = s
    time.sleep(0.4)   # pausa cortés para no saturar el servidor

print('\n✅ Descarga completada.')

In [ ]:
# ─── 5. Construcción del DataFrame consolidado ───────────────────────────────

df = pd.DataFrame(all_series)
df.index.name = 'Año'
df.index = df.index.astype(int)
df = df.sort_index()

# Vista previa
print(f'Dimensiones: {df.shape[0]} años × {df.shape[1]} series')
df.tail(10)

In [ ]:
# ─── 6. Exportar a datos.xlsx como nueva hoja ─────────────────────────────────

SHEET_NAME = 'C-4_Historico'

if DATA_PATH.exists():
    # Abrir el workbook existente y agregar / reemplazar la hoja
    with pd.ExcelWriter(DATA_PATH, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=SHEET_NAME)
    print(f'✅ Hoja "{SHEET_NAME}" guardada en {DATA_PATH}')
else:
    # Crear el archivo si no existe
    df.to_excel(DATA_PATH, sheet_name=SHEET_NAME)
    print(f'✅ Archivo creado: {DATA_PATH}')

In [ ]:
# ─── 7. Visualizaciones rápidas ───────────────────────────────────────────────
# Tres gráficos clave para una primera lectura de los datos

fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=False)
fig.suptitle('Finanzas Públicas del Perú — Series Históricas (% del PBI)\nFuente: BCRPData', 
             fontsize=13, y=0.98)

# ── Gráfico 1: Resultado Económico y Primario ──
ax1 = axes[0]
df['Resultado Económico SPNF'].dropna().plot(ax=ax1, color='#e74c3c', linewidth=2, label='Resultado Económico')
df['Resultado Primario SPNF'].dropna().plot(ax=ax1, color='#3498db', linewidth=2, linestyle='--', label='Resultado Primario')
ax1.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax1.fill_between(df.index, df['Resultado Económico SPNF'].dropna().reindex(df.index), 0,
                 where=df['Resultado Económico SPNF'].notna(),
                 alpha=0.08, color='#e74c3c')
ax1.set_title('Resultado Fiscal del SPNF')
ax1.set_ylabel('% del PBI')
ax1.legend()
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax1.grid(axis='y', alpha=0.4)

# ── Gráfico 2: Composición de Ingresos GC ──
ax2 = axes[1]
df['Total Ingresos Corrientes GC'].dropna().plot(ax=ax2, color='#2ecc71', linewidth=2.5, label='Total Ingresos Corrientes')
df['Ingresos Tributarios GC'].dropna().plot(ax=ax2, color='#27ae60', linewidth=1.5, linestyle='--', label='Ing. Tributarios')
df['Impuesto a la Renta (IR)'].dropna().plot(ax=ax2, color='#f39c12', linewidth=1.5, label='IR')
df['IGV'].dropna().plot(ax=ax2, color='#8e44ad', linewidth=1.5, label='IGV')
ax2.set_title('Ingresos del Gobierno Central')
ax2.set_ylabel('% del PBI')
ax2.legend(fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax2.grid(axis='y', alpha=0.4)

# ── Gráfico 3: Deuda Pública (desde 1999) ──
ax3 = axes[2]
df_deuda = df[['Deuda Pública Total', 'Deuda Pública Externa', 'Deuda Pública Interna']].dropna(how='all')
df_deuda['Deuda Pública Total'].plot(ax=ax3, color='#e74c3c', linewidth=2.5, label='Total')
df_deuda['Deuda Pública Externa'].plot(ax=ax3, color='#e67e22', linewidth=1.5, linestyle='--', label='Externa')
df_deuda['Deuda Pública Interna'].plot(ax=ax3, color='#2980b9', linewidth=1.5, linestyle='--', label='Interna')
ax3.set_title('Saldo de Deuda Pública del SPNF (desde 1999)')
ax3.set_ylabel('% del PBI')
ax3.set_xlabel('Año')
ax3.legend()
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax3.grid(axis='y', alpha=0.4)

plt.tight_layout(rect=[0, 0, 1, 0.97])

# Guardar figura
fig_path = Path('../data/finanzas_publicas_historico.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico guardado en {fig_path}')

In [ ]:
# ─── 8. Estadísticas descriptivas básicas ─────────────────────────────────────
# Útil para que los alumnos calibren valores típicos antes de analizar el MMM

print('=' * 65)
print(f'ESTADÍSTICAS DESCRIPTIVAS — {YEAR_INI}–{YEAR_FIN}')
print('=' * 65)
df.describe().round(2)

---
## Nota metodológica

- Todas las series están en **% del PBI**.
- Las series de ingresos corresponden al **Gobierno Central** (no al SPNF) por disponibilidad de la desagregación tributaria.
- Los datos de deuda están disponibles **desde 1999** solamente.
- Las cifras de 2025 son preliminares según el BCRP.
- Para cambiar el período de consulta, modificar `YEAR_INI` y `YEAR_FIN` en la celda de configuración.

**Códigos de series usados — referencia rápida:**

```
PM05780FA  Resultado Económico SPNF (% PBI)
PM05776FA  Resultado Primario SPNF (% PBI)
PM10083FA  Intereses SPNF (% PBI)
PM10080FA  Gastos de Capital SPNF (% PBI)
PM10081FA  Inversión Pública SPNF (% PBI)
PM10117FA  Total Ingresos Corrientes GC (% PBI)
PM10103FA  Ingresos Tributarios GC (% PBI)
PM10104FA  Impuesto a la Renta (% PBI)
PM10108FA  IGV (% PBI)
PM10111FA  ISC (% PBI)
PM10116FA  Ingresos No Tributarios GC (% PBI)
PM10188FA  Deuda Pública Total (% PBI)  — desde 1999
PM10189FA  Deuda Pública Externa (% PBI) — desde 1999
PM10197FA  Deuda Pública Interna (% PBI) — desde 1999
PM10213FA  Bonos Soberanos (% PBI)       — desde 1999
```